# Generate CRML models from natural-language requirements

For each domain (SRI, Traffic, Pumps) this notebook iterates over the
requirements defined in `experiments.tests.TESTS` and generates each
requirement **individually** from the domain seed model (no multi-turn
sequences). Generated `.crml` files are written to `generated/`.

Utility functions live in `experiments.util`:
- `extract_crml_block`   — pull a CRML code block from LLM output
- `generate_crml_single` — single-requirement LLM conversation from seed

In [ ]:
import json
import os
from pathlib import Path

from utils import MultiAgent, create_backend
from experiments.tests import TESTS
from experiments.util import generate_crml_single, Tee

# ── Config ────────────────────────────────────────────────────────────────────
LOCAL = False

if LOCAL:
    OLLAMA_HOST  = "http://127.0.0.1:11434"
    AUTH         = {}
else:
    OLLAMA_HOST  = "https://demo.narancsle.cc"
    AUTH = {
        "CF-Access-Client-Id":     os.environ.get("CF_Access_Client_Id"),
        "CF-Access-Client-Secret": os.environ.get("CF_Access_Client_Secret"),
    }

MCP_CRML_URL = "https://crml-mcp.narancsle.cc/mcp"
K = 5   # number of independent runs per requirement (best-of-k)


async def _try_backend(*args, **kwargs):
    try:
        return await create_backend(*args, **kwargs)
    except Exception as e:
        print(f"Backend unavailable ({args[0]}/{args[1]}): {e}")
        return None


OPTIONS = {
    "qwen3.5:27b": {
        "folder": "generated/qwen3.5_27b/",
        "endpoint": await _try_backend("ollama", "qwen3.5:27b", host=OLLAMA_HOST, headers=AUTH),
    },
    "claude": {
        "folder": "generated/claude/",
        "endpoint": await _try_backend("anthropic", "claude-sonnet-4-5", api_key=os.environ.get("ANTHROPIC_API_KEY")),
    },
    "openai": {
        "folder": "generated/openai/",
        "endpoint": await _try_backend("openai", "gpt-5-mini-2025-08-07", api_key=os.environ.get("OPENAI_API_KEY")),
    },
}
# ─────────────────────────────────────────────────────────────────────────────

settings = OPTIONS["qwen3.5:27b"]

backend    = settings["endpoint"]
OUTPUT_DIR = Path(settings["folder"])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## SRI domain

In [ ]:
DOMAIN = "SRI"
seed   = TESTS[DOMAIN]["seed"]

for req in TESTS[DOMAIN]["requirements"]:
    req_id   = req["id"]
    req_text = req["en"]

    print(f"\n{'='*100}\n{DOMAIN} / {req_id}\n{'='*100}")
    print(f"  {req_text}\n")

    for k in range(1, K + 1):
        print(f"\n--- run {k}/{K} ---")
        out_path = OUTPUT_DIR / f"{DOMAIN}_{req_id}_k{k}.crml"
        with Tee(out_path.with_suffix(".log")):
            async with MultiAgent([MCP_CRML_URL], backend) as agent:
                result  = await generate_crml_single(agent, seed, req_text)
                metrics = agent.cumulative_metrics

        out_path.write_text(result)
        out_path.with_suffix(".json").write_text(json.dumps(metrics, indent=2))
        print(f"Saved → {out_path}")

## Traffic-light domain

In [ ]:
DOMAIN = "traffic"
seed   = TESTS[DOMAIN]["seed"]

for req in TESTS[DOMAIN]["requirements"]:
    req_id   = req["id"]
    req_text = req["en"]

    print(f"\n{'='*100}\n{DOMAIN} / {req_id}\n{'='*100}")
    print(f"  {req_text}\n")

    for k in range(1, K + 1):
        print(f"\n--- run {k}/{K} ---")
        out_path = OUTPUT_DIR / f"{DOMAIN}_{req_id}_k{k}.crml"
        with Tee(out_path.with_suffix(".log")):
            async with MultiAgent([MCP_CRML_URL], backend) as agent:
                result  = await generate_crml_single(agent, seed, req_text)
                metrics = agent.cumulative_metrics

        out_path.write_text(result)
        out_path.with_suffix(".json").write_text(json.dumps(metrics, indent=2))
        print(f"Saved → {out_path}")

## Pumping-system domain

In [ ]:
DOMAIN = "pumpsystem"
seed   = TESTS[DOMAIN]["seed"]

for req in TESTS[DOMAIN]["requirements"]:
    req_id   = req["id"]
    req_text = req["en"]

    print(f"\n{'='*100}\n{DOMAIN} / {req_id}\n{'='*100}")
    print(f"  {req_text}\n")

    for k in range(1, K + 1):
        print(f"\n--- run {k}/{K} ---")
        out_path = OUTPUT_DIR / f"{DOMAIN}_{req_id}_k{k}.crml"
        with Tee(out_path.with_suffix(".log")):
            async with MultiAgent([MCP_CRML_URL], backend) as agent:
                result  = await generate_crml_single(agent, seed, req_text)
                metrics = agent.cumulative_metrics

        out_path.write_text(result)
        out_path.with_suffix(".json").write_text(json.dumps(metrics, indent=2))
        print(f"Saved → {out_path}")